# Benchmark summary comparison

In [ ]:
import sys
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PYTHON_SRC = Path.cwd() / "python"
if str(PYTHON_SRC) not in sys.path:
    sys.path.insert(0, str(PYTHON_SRC))

import benchmark_reporting.constants as constants
import benchmark_reporting.plotting as plotting
import benchmark_reporting.comparison as comparison

importlib.reload(constants)
importlib.reload(plotting)
importlib.reload(comparison)

from benchmark_reporting.constants import *
from benchmark_reporting.comparison import *

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.4g}".format)

In [ ]:
BASELINE_SUMMARY_JSON = Path("benchmark_summary_baseline.json")
CANDIDATE_SUMMARY_JSON = Path("benchmark_summary_candidate.json")

SELECTED_REFERENCE_KEY = "sklearn_brute"
REFERENCE_FALLBACK_KEY = "reference"

SPEEDUP_DETAIL_LIMIT = 30

CACHEGRIND_DETAIL_LIMIT = 30
CACHEGRIND_HEATMAP_COUNTERS = [COL_CACHEGRIND_IR, COL_CACHEGRIND_D1MR]

PARITY_DETAIL_LIMIT = 30
PARITY_NUMERIC_DELTA_LIMIT = 30
SPILL_DETAIL_LIMIT = 30


## Load summaries

In [ ]:
baseline = load_labeled_summary(BASELINE_SUMMARY_JSON)
candidate = load_labeled_summary(CANDIDATE_SUMMARY_JSON)

compatibility = compare_summary_compatibility(
    baseline,
    candidate,
    selected_reference_key=SELECTED_REFERENCE_KEY,
    fallback_reference_key=REFERENCE_FALLBACK_KEY,
)

display(Markdown(
    "### Inferred run labels\n\n"
    f"- **Baseline:** `{baseline.label}`\n"
    f"- **Candidate:** `{candidate.label}`"
))

## Summary compatibility

In [ ]:
display(Markdown("### Overview"))
display(compatibility["overview"])

display(Markdown("### Reference-key resolution"))
display(compatibility["reference_resolution"])

display(Markdown("### Strict point-set counts"))
display(compatibility["point_counts"])

## Build and environment context

In [ ]:
display(display_wrapped_dataframe(compatibility["run_context"]))

## Missing strict-match structure

In [ ]:
missing_hierarchy = compatibility["missing_hierarchy"]

if missing_hierarchy.empty:
    display(Markdown("No one-sided strict-match points."))
else:
    display(Markdown(
        "Rows describe the highest useful missing structure. "
        "`Structural Summary` shows varying non-grid values below the path; "
        "`Grid Summary` shows the affected `D`/`N`/`K` slice."
    ))
    display_cols = [
        COL_MISSING_LEVEL,
        COL_PATH,
        COL_MISSING_POINTS,
        COL_STRUCTURAL_SUMMARY,
        COL_GRID_SUMMARY,
        COL_MISSING_PATTERN,
        COL_NEXT_DETAIL,
    ]
    display_cols = [col for col in display_cols if col in missing_hierarchy.columns]

    for point_set in missing_hierarchy[COL_POINT_SET].drop_duplicates():
        point_set_df = missing_hierarchy[missing_hierarchy[COL_POINT_SET] == point_set]
        display(Markdown(f"### {point_set}"))
        for direction in point_set_df[COL_DIRECTION].drop_duplicates():
            direction_df = point_set_df[point_set_df[COL_DIRECTION] == direction]
            if direction_df.empty:
                continue
            display(Markdown(f"**{direction}**"))
            display(display_wrapped_dataframe(direction_df[display_cols].reset_index(drop=True)))


## Speedup comparison

This section uses only exact matched points. The metric is:

```text
100 * (candidate_speedup / baseline_speedup - 1)
```

Positive values mean the candidate run has higher Python/C++ speedup than the baseline run.

In [ ]:
df_speedup_comparison = make_speedup_comparison(
    baseline.speedups,
    candidate.speedups,
    selected_reference_key=SELECTED_REFERENCE_KEY,
    fallback_reference_key=REFERENCE_FALLBACK_KEY,
)

resolved_reference_key = resolve_reference_key(
    baseline.speedups,
    candidate.speedups,
    selected_key=SELECTED_REFERENCE_KEY,
    fallback_key=REFERENCE_FALLBACK_KEY,
)

if df_speedup_comparison.empty:
    display(Markdown(
        f"No exact matched speedup rows for reference key `{resolved_reference_key}`."
    ))
else:
    display(Markdown(
        f"Matched speedup rows: **{len(df_speedup_comparison)}**  \n"
        f"Reference key: `{resolved_reference_key}`"
    ))

    summaries = summarize_speedup_comparison_default_groups(df_speedup_comparison)

    display(Markdown("### Speedup-delta summary by phase"))
    display(summaries["by_phase"])

    display(Markdown("### Speedup-delta summary by phase/stage"))
    display(summaries["by_phase_stage"])

    display(Markdown("### Speedup-delta summary by phase/stage/variant"))
    display(summaries["by_phase_stage_variant"])

    display(Markdown("### Speedup-delta summary by phase/stage/variant/params"))
    display(summaries["by_phase_stage_variant_params"])

    display(Markdown("### Speedup-delta summary by D, N, and K"))
    display(Markdown("#### By D"))
    display(summaries["by_dimensions"])
    display(Markdown("#### By N"))
    display(summaries["by_samples"])
    display(Markdown("#### By K"))
    display(summaries["by_clusters"])

    display(Markdown(
        f"### Strict matched speedup rows, top {SPEEDUP_DETAIL_LIMIT} by absolute speedup change"
    ))

    detail_cols = speedup_comparison_display_columns(df_speedup_comparison)

    df_speedup_detail = (
        df_speedup_comparison
        .assign(
            abs_speedup_delta_pct=lambda df: df[COL_SPEEDUP_DELTA_PCT].abs()
        )
        .sort_values(
            ["abs_speedup_delta_pct", COL_SPEEDUP_RATIO],
            ascending=[False, False],
            na_position="last",
        )
    )

    display(df_speedup_detail[detail_cols].head(SPEEDUP_DETAIL_LIMIT))

## Speedup-delta heatmap

In [ ]:
if df_speedup_comparison.empty:
    display(Markdown("No exact matched speedup rows to plot."))
else:
    plot_clustered_delta_heatmaps(
        comparison_df=df_speedup_comparison,
        value_col=COL_SPEEDUP_DELTA_PCT,
        group_cols=[
            COL_PHASE,
            COL_STAGE,
            COL_VARIANT,
            COL_PARAMS,
            COL_REFERENCE_KEY,
        ],
        title_prefix="Speedup Delta Heatmap by K",
        cbar_label="Speedup delta (%)",
        positive_meaning="positive means candidate speedup increased",
    )


## Cachegrind comparison

This section compares raw Cachegrind counters only. There is no per-sample or per-algorithm-iteration normalization yet.

Positive deltas mean the candidate has a larger raw counter value than the baseline.

In [ ]:
df_cachegrind_comparison = make_cachegrind_comparison(
    baseline.cachegrind,
    candidate.cachegrind,
)

if df_cachegrind_comparison.empty:
    display(Markdown("No exact matched Cachegrind rows."))
else:
    available_counters = cachegrind_available_counters(df_cachegrind_comparison)
    display(Markdown(
        f"Matched Cachegrind rows: **{len(df_cachegrind_comparison)}**  \n"
        f"Counters: `{', '.join(available_counters)}`"
    ))

    display(Markdown("### Raw counter-delta summary by phase/stage/counter"))
    display(summarize_cachegrind_comparison(
        df_cachegrind_comparison,
        group_cols=[COL_PHASE, COL_STAGE, COL_COUNTER],
    ))

    display(Markdown("### Raw counter-delta summary by phase/stage/variant/counter"))
    display(summarize_cachegrind_comparison(
        df_cachegrind_comparison,
        group_cols=[COL_PHASE, COL_STAGE, COL_VARIANT, COL_COUNTER],
    ))

    display(Markdown(f"### Strict matched Cachegrind rows, first {CACHEGRIND_DETAIL_LIMIT}"))
    detail_cols = cachegrind_comparison_display_columns(df_cachegrind_comparison)
    display(df_cachegrind_comparison[detail_cols].head(CACHEGRIND_DETAIL_LIMIT))

## Cachegrind counter heatmaps

In [ ]:
if df_cachegrind_comparison.empty:
    display(Markdown("No exact matched Cachegrind rows to plot."))
else:
    available_counters = cachegrind_available_counters(df_cachegrind_comparison)
    for counter in CACHEGRIND_HEATMAP_COUNTERS:
        if counter not in available_counters:
            display(Markdown(f"Skipping `{counter}`: counter not present in both summaries."))
            continue

        plot_clustered_delta_heatmaps(
            df_cachegrind_comparison,
            value_col=cachegrind_counter_delta_pct_col(counter),
            group_cols=[COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS],
            title_prefix=f"Cachegrind {counter} Delta Heatmap by K",
            cbar_label=f"{counter} delta (%)",
            positive_meaning="positive means raw counter increased",
        )


## Parity comparison

This section compares parity status transitions and shared numeric parity metrics.

In [ ]:
df_parity_comparison = make_parity_comparison(
    baseline.parity,
    candidate.parity,
    selected_reference_key=SELECTED_REFERENCE_KEY,
    fallback_reference_key=REFERENCE_FALLBACK_KEY,
)

if df_parity_comparison.empty:
    display(Markdown("No exact matched parity rows."))
else:
    df_parity_status_transitions = filter_parity_status_transitions(df_parity_comparison)

    display(Markdown(
        f"Matched parity rows: **{len(df_parity_comparison)}**  \n"
        f"Rows with real status transitions: **{len(df_parity_status_transitions)}**"
    ))

    display(Markdown("### Status transitions by phase/stage/variant/params"))
    status_transition_summary = summarize_parity_transitions(
        df_parity_comparison,
        group_cols=[COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS, COL_STATUS_TRANSITION],
    )
    if status_transition_summary.empty:
        display(Markdown("No parity status changed between the two summaries."))
    else:
        display(status_transition_summary)

    display(Markdown("### Numeric parity-delta summary by phase/stage/params/metric"))
    numeric_delta_summary = summarize_parity_numeric_deltas(
        df_parity_comparison,
        group_cols=[COL_PHASE, COL_STAGE, COL_PARAMS, COL_PARITY_NUMERIC_METRIC],
    )
    if numeric_delta_summary.empty:
        display(Markdown("No shared numeric parity metrics to compare."))
    else:
        display(numeric_delta_summary.head(PARITY_NUMERIC_DELTA_LIMIT))

    df_parity_pressure_detail = sort_parity_by_pressure_delta(
        df_parity_comparison,
        transitions_only=True,
    )

    if df_parity_pressure_detail.empty:
        display(Markdown("No strict matched parity rows with a status transition."))
    else:
        display(Markdown(
            f"### Strict matched parity rows with status transitions, "
            f"top {PARITY_DETAIL_LIMIT} by absolute parity-pressure change"
        ))
        detail_cols = parity_comparison_display_columns(df_parity_pressure_detail)
        display(df_parity_pressure_detail[detail_cols].head(PARITY_DETAIL_LIMIT))


## Spill-detection comparison

This section compares spill-detector candidate reload-pair counts. Positive deltas mean the candidate run reports more reload pairs than the baseline run.


In [ ]:
df_spill_comparison = make_spill_comparison(
    baseline.spills,
    candidate.spills,
)

if df_spill_comparison.empty:
    display(Markdown("No spill-detection rows."))
else:
    display(Markdown(f"Spill-detection rows: **{len(df_spill_comparison)}**"))

    display(Markdown("### Strict point matching"))
    spill_match_counts = (
        df_spill_comparison["_merge"]
        .value_counts()
        .rename_axis("Match Status")
        .reset_index(name="Rows")
    )
    display(spill_match_counts)

    matched_spills = df_spill_comparison[df_spill_comparison["_merge"] == "both"].copy()
    if matched_spills.empty:
        display(Markdown("No exact matched spill-detection rows."))
    else:
        matched_spills["Reload Pair Abs Delta"] = matched_spills["Reload Pair Delta"].abs()

        display(Markdown("### Reload-pair delta summary by phase/stage/variant/params"))
        display(
            matched_spills
            .groupby([COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS], observed=True, dropna=False)
            .agg(
                **{
                    COL_MATCHED_POINTS: ("Reload Pair Delta", "size"),
                    "Median Reload Pair Delta": ("Reload Pair Delta", "median"),
                    "Max Reload Pair Abs Delta": ("Reload Pair Abs Delta", "max"),
                }
            )
            .reset_index()
            .sort_values([COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS])
        )

        display(Markdown(f"### Strict matched spill-detection rows, top {SPILL_DETAIL_LIMIT} by absolute reload-pair change"))
        spill_detail_cols = SPILL_MATCH_COLS + [
            "Candidate Reload Pairs_baseline",
            "Candidate Reload Pairs_candidate",
            "Reload Pair Delta",
        ]
        display(
            matched_spills
            .sort_values("Reload Pair Abs Delta", ascending=False, na_position="last")
            [spill_detail_cols]
            .head(SPILL_DETAIL_LIMIT)
        )

    one_sided_spills = df_spill_comparison[df_spill_comparison["_merge"] != "both"].copy()
    if not one_sided_spills.empty:
        display(Markdown(f"### One-sided spill-detection rows, first {SPILL_DETAIL_LIMIT}"))
        one_sided_cols = SPILL_MATCH_COLS + [
            "_merge",
            "Candidate Reload Pairs_baseline",
            "Candidate Reload Pairs_candidate",
        ]
        display(one_sided_spills[one_sided_cols].head(SPILL_DETAIL_LIMIT))
